# Chapter 1 — Introduction to Attention and Transformers

This notebook builds intuition for *why* transformers use attention and *what* the basic
encoder and decoder layer look like. We use PyTorch's built-in `nn.MultiheadAttention`
as a placeholder — later chapters (07–09) implement MHA from scratch.

## What is Attention?

In a traditional RNN, every token in a sequence passes information forward one step at a
time. By the time token 50 needs context from token 1, the signal has been compressed
through 49 intermediate states. **Attention** is the alternative: let every token
directly query every other token and decide how much to borrow from each.

Concretely, given a sequence of embeddings with shape `(batch, seq_len, d_model)`, the
attention operation:

1. Projects each embedding into three vectors: **query**, **key**, **value**.
2. Computes pairwise similarity scores: `Q @ Kᵀ / sqrt(d_k)`.
3. Normalises with softmax → attention weights (a probability distribution over positions).
4. Returns a weighted sum of the value vectors.

The output is still `(batch, seq_len, d_model)` — same shape in, same shape out.

## What is a Transformer Block?

A transformer encoder layer wraps attention with two architectural patterns:

- **Residual connection**: `x = x + sublayer(x)` — lets gradients flow unchanged to
  early layers during training.
- **Layer normalisation**: stabilises activations across the `d_model` dimension.

The full encoder layer applies two such sublayers in sequence:
1. Self-attention (each token attends to all tokens in the same sequence).
2. Feed-forward network (two linear layers with ReLU, applied per-token independently).


In [1]:
import torch
from transformer_book_lab.encoder_layer import EncoderLayer

# Post-norm EncoderLayer (original 'Attention Is All You Need' style)
layer = EncoderLayer(d_model=64, nhead=4, dim_feedforward=128, pre_norm=False)
layer.eval()

x = torch.randn(1, 6, 64)  # (batch=1, seq_len=6, d_model=64)
with torch.no_grad():
    out = layer(x)

print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")
print("Shape is preserved:", x.shape == out.shape)


Input shape:  torch.Size([1, 6, 64])
Output shape: torch.Size([1, 6, 64])
Shape is preserved: True


## Post-Norm vs. Pre-Norm

Where you place LayerNorm relative to the residual connection affects training stability.

**Post-norm** (original Transformer, 2017):
```
x  → [Self-Attention] → x + attn_out → [LayerNorm] → x'
x' → [Feed-Forward]  → x' + ff_out  → [LayerNorm] → output
```
The residual is added *before* normalisation. Gradients can grow large early in training
because the unnormalised sum can be large.

**Pre-norm** (modern default — GPT-2, LLaMA, etc.):
```
x  → [LayerNorm] → [Self-Attention] → x + attn_out → x'
x' → [LayerNorm] → [Feed-Forward]  → x' + ff_out  → output
```
The input is normalised *before* the sublayer sees it. The residual path is always the
raw `x`, which gives cleaner gradient flow and allows training without a warm-up schedule.

Our `EncoderLayer` supports both via the `pre_norm` flag.


In [2]:
# Compare post-norm vs. pre-norm outputs on the same input
torch.manual_seed(42)
x = torch.randn(1, 6, 64)

post_norm_layer = EncoderLayer(d_model=64, nhead=4, dim_feedforward=128, pre_norm=False)
pre_norm_layer  = EncoderLayer(d_model=64, nhead=4, dim_feedforward=128, pre_norm=True)
post_norm_layer.eval()
pre_norm_layer.eval()

with torch.no_grad():
    out_post = post_norm_layer(x)
    out_pre  = pre_norm_layer(x)

print(f"Post-norm output shape: {out_post.shape}")
print(f"Pre-norm  output shape: {out_pre.shape}")
print(f"Outputs differ (random weights): {not torch.allclose(out_post, out_pre)}")


Post-norm output shape: torch.Size([1, 6, 64])
Pre-norm  output shape: torch.Size([1, 6, 64])
Outputs differ (random weights): True


## The Encoder-Decoder Architecture

The original transformer has two halves:

**Encoder**: reads the source sequence (e.g. a French sentence) and produces a sequence
of continuous representations called *memory*. Each encoder layer applies self-attention
(every source token attends to every other source token) followed by a feed-forward
network. The encoder is **bidirectional** — no masking, full context in both directions.

**Decoder**: generates the target sequence (e.g. an English translation) one token at a
time. Each decoder layer has three sublayers:
1. **Causal self-attention**: each generated token attends only to previously generated
   tokens (future positions are masked out).
2. **Cross-attention**: queries come from the decoder, keys and values come from the
   encoder memory — this is how the decoder *reads* the source.
3. **Feed-forward network**: same per-token MLP as in the encoder.

```
Source tokens
     │
  [Encoder Stack]        ← self-attention only (bidirectional)
     │ memory
     ├──────────────────┬
     │                  │
  Target tokens     [Decoder Stack]  ← causal self-attn + cross-attn
                         │
                    Target output
```

Memory flows from the encoder to every decoder layer via cross-attention. The decoder
sees the entire encoder output at every decoding step.


In [3]:
from transformer_book_lab.decoder_layer import DecoderLayer

decoder = DecoderLayer(d_model=64, nhead=4, dim_feedforward=128)
decoder.eval()

# Simulate: encoder produced 6-token memory, decoder generates a 4-token target
memory = torch.randn(1, 6, 64)  # (batch=1, src_len=6, d_model=64)
tgt    = torch.randn(1, 4, 64)  # (batch=1, tgt_len=4, d_model=64)

with torch.no_grad():
    out = decoder(tgt, memory)

print(f"Memory shape: {memory.shape}")
print(f"Target shape: {tgt.shape}")
print(f"Output shape: {out.shape}")
print("Output shape matches target:", out.shape == tgt.shape)


Memory shape: torch.Size([1, 6, 64])
Target shape: torch.Size([1, 4, 64])
Output shape: torch.Size([1, 4, 64])
Output shape matches target: True


## What I Learned

Key takeaways from this chapter:

- **Attention replaces recurrence**: every token can directly query every other token,
  so long-range dependencies cost the same as short-range ones.
- **Residual connections are load-bearing**: without them, deeper stacks do not train well.
  The residual path carries the original signal; each sublayer learns a *correction*.
- **Pre-norm vs. post-norm is a placement detail**: the maths is the same, but pre-norm
  puts the normalised representation into the sublayer, making training more stable.
- **The encoder is fully bidirectional**: it can see the whole source at once.
  The decoder is causal: it can only see past generated tokens, not future ones.
- **Cross-attention is the bridge**: the decoder reads the encoder’s memory through
  attention — the target generates queries, the encoder provides keys and values.


In [4]:
# Reflection: inspect the parameter counts of each layer type
import torch.nn as nn

enc = EncoderLayer(d_model=64, nhead=4, dim_feedforward=128)
dec = DecoderLayer(d_model=64, nhead=4, dim_feedforward=128)

enc_params = sum(p.numel() for p in enc.parameters())
dec_params = sum(p.numel() for p in dec.parameters())

print(f"EncoderLayer parameters: {enc_params:,}")
print(f"DecoderLayer parameters: {dec_params:,}")
print(f"Decoder has ~{dec_params / enc_params:.1f}x more params (extra cross-attn block)")


EncoderLayer parameters: 33,472
DecoderLayer parameters: 50,240
Decoder has ~1.5x more params (extra cross-attn block)


## Final Exercise

Stack two `EncoderLayer`s and pass a sequence through. Verify the output shape is still
`(batch, seq_len, d_model)` after two layers.

**Why this matters**: stacking N encoder layers is exactly what BERT and other
encoder-only transformers do. Each layer refines the contextual representation.


In [5]:
# Final exercise: stack two EncoderLayers

class StackedEncoder(nn.Module):
    def __init__(self, d_model: int, nhead: int, dim_feedforward: int) -> None:
        super().__init__()
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, nhead, dim_feedforward, pre_norm=True),
            EncoderLayer(d_model, nhead, dim_feedforward, pre_norm=True),
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return x


stacked = StackedEncoder(d_model=64, nhead=4, dim_feedforward=128)
stacked.eval()

x = torch.randn(2, 10, 64)  # (batch=2, seq_len=10, d_model=64)
with torch.no_grad():
    out = stacked(x)

print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")
assert out.shape == x.shape, "Shape must be preserved through both layers!"
print("Exercise complete: two stacked EncoderLayers preserve shape.")


Input shape:  torch.Size([2, 10, 64])
Output shape: torch.Size([2, 10, 64])
Exercise complete: two stacked EncoderLayers preserve shape.
